# Aeon Ephys Pipeline Guide

End-to-end guide for processing ephys data through the Aeon pipeline.

**Prerequisites:** Complete the setup in [step00_setup.md](step00_setup.md) before running this notebook.

This notebook is a thin wrapper around the guide scripts (`step01_*.py` through `step06_*.py`). Each cell calls one function from a script. The scripts contain all the logic and inline documentation — open them to understand what each step does.

In [ ]:
# Add the guide directory to sys.path so imports work regardless of
# where the notebook is launched from.
import sys
from pathlib import Path

guide_dir = Path("docs/ephys_guide").resolve()
if str(guide_dir) not in sys.path:
    sys.path.insert(0, str(guide_dir))

## Configuration

Edit these variables for your experiment. All subsequent cells use these values.

In [ ]:
# ---------- Edit these for your experiment ----------
experiment_name = "abcEphysPilot02-aeonx1"
raw_data_dir = "/ceph/aeon/aeon/data/raw/AEONX1/abcEphysPilot02"
subject = "BAA-XXXXXXX"  # Replace with actual SWC subject ID
# DB prefix is configured in datajoint.json (or DJ_DATABASE_PREFIX env var),
# not in this notebook. See step00_setup.md for details.

# Probe info
probe_serial = "23299108854"
probe_type = "neuropixels2.0"
channel_map_file = "recording_configurations/M81_ProbeB_4Shanks_1000_to_1700_um.json"

# Block params (test subset)
block_duration_min = 30
overlap_min = 10
n_blocks = 3

# Sorting params
paramset_id = 400
sorting_method = "kilosort4"
electrode_group_name = "all"
matching_paramset_id = 1

## Step 1: Register Experiment

Register the experiment, subject, probe type, electrode configuration, and probe assignments. Then ingest epochs and chunks.

In [ ]:
from step01_register_experiment import (
    register_experiment,
    ensure_probe_type,
    create_electrode_config,
    create_probe_assignments,
    register_epochs,
    ingest_chunks,
    verify_registration,
)

register_experiment(experiment_name, raw_data_dir, subject)
ensure_probe_type(probe_type)
create_electrode_config(raw_data_dir, channel_map_file, probe_type)
create_probe_assignments(raw_data_dir, probe_serial, subject)
register_epochs(experiment_name)
ingest_chunks(experiment_name)
verify_registration(experiment_name)

## Step 2: Define Blocks

Create time windows for spike sorting. Test configuration: 3 blocks of 30 minutes with 10-minute overlap.

In [ ]:
from step02_define_blocks import query_available_data, create_blocks, verify_blocks

query_available_data(experiment_name)
create_blocks(experiment_name, subject, block_duration_min, overlap_min, n_blocks)
verify_blocks(experiment_name)

## Step 3: Spike Sorting

Set up sorting prerequisites, run preprocessing, and get SLURM configuration for GPU sorting.

In [ ]:
from step03_spike_sorting import (
    setup_sorting_prerequisites,
    run_preprocessing,
    print_slurm_config,
)

setup_sorting_prerequisites(
    experiment_name, subject, paramset_id, sorting_method, electrode_group_name
)
run_preprocessing(experiment_name)
print_slurm_config(experiment_name, subject)

### STOP: Submit the SLURM job

Copy the SLURM configuration printed above, submit the sorting job with `sbatch`, and wait for it to complete before continuing.

In [ ]:
# Run this cell AFTER the SLURM sorting job has completed.
from step03_spike_sorting import run_post_sorting, verify_sorting

run_post_sorting(experiment_name)
verify_sorting(experiment_name)

## Step 4: Curation

See [step04_curation.md](step04_curation.md) for curation instructions. Use Path A (auto-approval) for testing, or Path B (manual curation with SpikeInterface GUI) for production.

The cell below runs auto-approval (Path A). Skip it if you plan to curate manually.

In [ ]:
# Auto-approval (Path A) -- skip if doing manual curation
from datetime import datetime, timezone
from aeon.dj_pipeline import spike_sorting
from aeon.dj_pipeline import spike_sorting_curation as curation

if not (curation.CurationMethod & {"curation_method": "SpikeInterface"}):
    curation.CurationMethod.insert1({"curation_method": "SpikeInterface"}, skip_duplicates=True)

sorting_entries = (spike_sorting.SpikeSorting & {"experiment_name": experiment_name}).keys()
now = datetime.now(timezone.utc)
for sorting_key in sorting_entries:
    mc_key = {**sorting_key, "curation_id": 0}
    if not (curation.ManualCuration & mc_key):
        curation.ManualCuration.insert1({
            **mc_key, "curation_datetime": now, "parent_curation_id": -1,
            "curation_method": "SpikeInterface",
            "description": "Auto-approved: no manual curation applied",
        }, skip_duplicates=True)
    sorted_pk = {k: sorting_key[k] for k in spike_sorting.SortedSpikes.primary_key}
    if not (curation.OfficialCuration & sorted_pk):
        curation.OfficialCuration.insert1({**sorted_pk, "curation_id": 0}, skip_duplicates=True)

curation.ApplyOfficialCuration.populate(display_progress=True)

## Step 5: Unit Matching

Sync spike times to the behavioral clock and match units across overlapping blocks.

In [ ]:
from step05_unit_matching import sync_spikes, run_unit_matching, verify_matching

sync_spikes(experiment_name)
run_unit_matching(experiment_name, subject, matching_paramset_id)
verify_matching(experiment_name)

## Step 6: Analysis Examples

Fetch spike times and create basic visualizations.

In [ ]:
from step06_analysis_examples import fetch_spike_times, plot_raster, fetch_behavioral_events

spike_dict = fetch_spike_times(experiment_name)
if spike_dict:
    plot_raster(spike_dict)
fetch_behavioral_events(experiment_name)

## Next Steps

- **Analysis:** See analysis repos for deeper analysis workflows
- **Production parameters:** Use 20-hour blocks with 5-hour overlap for real experiments
- **Manual curation:** Follow Path B in [step04_curation.md](step04_curation.md) for proper unit curation
- **Questions:** Contact the pipeline team